# Arabic Sign Language — Swin Transformer V2 Base Training
**Model:** Swin Transformer V2 Base (88M params) pretrained on ImageNet-22K

**Expected:** ~99.5% accuracy in ~30-45 min on Colab T4 GPU

### Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells
3. Download model from the Files panel (left sidebar) when done

In [ ]:
# Step 1: Install dependencies
!pip install -q timm

In [ ]:
# Step 2: Configure Kaggle API and download dataset
# Paste your Kaggle API token below
import os
os.environ['KAGGLE_API_TOKEN'] = 'YOUR_KAGGLE_TOKEN_HERE'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(os.environ['KAGGLE_API_TOKEN'])
os.chmod('/root/.kaggle/access_token', 0o600)
print('✓ Kaggle API configured')

In [ ]:
# Download and extract the dataset
!kaggle datasets download -d gannayasser/arabic-alphabets-sign-language-dataset-arasl -p /content/data --unzip
!ls /content/data/

# Find the actual data directory
import os
DATA_DIR = '/content/data/ArASL_Database_54K_Final/ArASL_Database_54K_Final'
if not os.path.isdir(DATA_DIR):
    # Try alternate path
    DATA_DIR = '/content/data/ArASL_Database_54K_Final'
if not os.path.isdir(DATA_DIR):
    for root, dirs, files in os.walk('/content/data'):
        if len(dirs) > 20:
            DATA_DIR = root
            break

print(f'Dataset dir: {DATA_DIR}')
print(f'Classes: {sorted(os.listdir(DATA_DIR))}')
print(f'Num classes: {len(os.listdir(DATA_DIR))}')

In [ ]:
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import numpy as np

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Config
IMG_SIZE = 256
BATCH_SIZE = 64
NUM_CLASSES = 32
NUM_WORKERS = 2
MODEL_NAME = 'swinv2_base_window12to16_192to256.ms_in22k_ft_in1k'

In [ ]:
# ── Data Loading with Augmentation ──
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label

full_dataset = datasets.ImageFolder(DATA_DIR)
n = len(full_dataset)
n_train = int(0.8 * n)
n_val = n - n_train
generator = torch.Generator().manual_seed(42)
train_idx, val_idx = torch.utils.data.random_split(range(n), [n_train, n_val], generator=generator)

train_dataset = TransformSubset(full_dataset, train_idx.indices, train_transform)
val_dataset = TransformSubset(full_dataset, val_idx.indices, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Classes ({len(full_dataset.classes)}): {full_dataset.classes}')
print(f'Total: {n} | Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
# ── Build Swin V2 Base ──
print(f'Loading {MODEL_NAME}...')
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,} ({total_params/1e6:.1f}M)')
print(f'Architecture: Swin Transformer V2 Base (ImageNet-22K pretrained)')

In [ ]:
# ── Training Functions ──
def freeze_base(model):
    for name, param in model.named_parameters():
        if 'head' not in name:
            param.requires_grad = False
    print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def unfreeze_top(model):
    all_p = list(model.named_parameters())
    cutoff = int(len(all_p) * 0.7)
    for i, (name, param) in enumerate(all_p):
        if i >= cutoff or 'head' in name:
            param.requires_grad = True
    print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

def train_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        if (i+1) % 100 == 0:
            print(f'  [{i+1}/{len(loader)}] Loss:{loss.item():.4f} Acc:{100.*correct/total:.1f}%')
    return loss_sum/total, 100.*correct/total

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast(device_type='cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return loss_sum/total, 100.*correct/total

In [ ]:
# ══════════════════════════════════════════════════════════
# PHASE 1: Transfer Learning (frozen base)
# ══════════════════════════════════════════════════════════
print('=' * 60)
print('PHASE 1: Transfer Learning (frozen base, lr=1e-3)')
print('=' * 60)

freeze_base(model)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
scaler = torch.amp.GradScaler()
best_acc = 0.0
patience_ctr = 0

for ep in range(1, 16):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler, ep)
    vl, va = validate(model, val_loader, criterion)
    scheduler.step()
    print(f'Ep {ep:2d} | Train:{ta:.2f}% | Val:{va:.2f}% | Loss:{vl:.4f} | {time.time()-t0:.0f}s')
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), '/content/asl_swinv2_best.pth')
        patience_ctr = 0
        print(f'  ★ Best: {va:.2f}%')
    else:
        patience_ctr += 1
        if patience_ctr >= 5:
            print(f'  Early stop'); break

print(f'\nPhase 1 best: {best_acc:.2f}%')

In [ ]:
# ══════════════════════════════════════════════════════════
# PHASE 2: Fine-tuning (top 30% unfrozen)
# ══════════════════════════════════════════════════════════
print('=' * 60)
print('PHASE 2: Fine-tuning (lr=1e-5)')
print('=' * 60)

model.load_state_dict(torch.load('/content/asl_swinv2_best.pth', map_location=DEVICE))
unfreeze_top(model)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = torch.amp.GradScaler()
patience_ctr = 0

for ep in range(1, 11):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader, criterion, optimizer, scaler, ep)
    vl, va = validate(model, val_loader, criterion)
    scheduler.step()
    print(f'Ep {ep:2d} | Train:{ta:.2f}% | Val:{va:.2f}% | Loss:{vl:.4f} | {time.time()-t0:.0f}s')
    if va > best_acc:
        best_acc = va
        torch.save(model.state_dict(), '/content/asl_swinv2_best.pth')
        patience_ctr = 0
        print(f'  ★ Best: {va:.2f}%')
    else:
        patience_ctr += 1
        if patience_ctr >= 5:
            print(f'  Early stop'); break

print(f'\n{"="*60}')
print(f'FINAL BEST ACCURACY: {best_acc:.2f}%')
print(f'{"="*60}')

In [ ]:
# ── Save metadata ──
meta = {
    'model_name': MODEL_NAME,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'classes': full_dataset.classes,
    'best_val_acc': best_acc,
    'normalize_mean': mean,
    'normalize_std': std,
}
with open('/content/asl_swinv2_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Model: /content/asl_swinv2_best.pth ({os.path.getsize("/content/asl_swinv2_best.pth")/1024/1024:.0f} MB)')
print(f'Meta:  /content/asl_swinv2_meta.json')
print(f'\n✅ Download both files → put in Arabic-Sign-Language/models/')

In [ ]:
# ── Download files ──
from google.colab import files
files.download('/content/asl_swinv2_best.pth')
files.download('/content/asl_swinv2_meta.json')

In [ ]:
# ── Per-class accuracy report ──
from collections import Counter
model.load_state_dict(torch.load('/content/asl_swinv2_best.pth', map_location=DEVICE))
model.eval()
correct_c, total_c = Counter(), Counter()
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        _, pred = model(imgs).max(1)
        for t, p in zip(labels, pred):
            total_c[t.item()] += 1
            if t == p: correct_c[t.item()] += 1

print(f'{"Class":<10} {"Acc":>8}')
print('-'*20)
for i in range(NUM_CLASSES):
    acc = 100*correct_c[i]/max(total_c[i],1)
    print(f'{full_dataset.classes[i]:<10} {acc:>7.2f}%')
print(f'\nOverall: {sum(correct_c.values())/sum(total_c.values())*100:.2f}%')